# 04 — The Precursor Test

**Do topological signatures systematically precede breakthroughs?**

This is the hypothesis test. For each cataloged breakthrough, we compare the
topological metrics (β₁, persistence entropy) in the years *before* the
breakthrough against a null model — the same CPC section pair at times when
no breakthrough occurred.

Key method: **Superposed epoch analysis** — align all breakthroughs at t=0
and average the signal. If topology is a precursor, we should see a systematic
rise before t=0 that the null model doesn't exhibit.

---

In [ ]:
# %% Imports and setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from tqdm import tqdm

from src.utils import DATA_DIR, get_logger
from src.breakthroughs import load_breakthroughs, get_precursor_window
from src.topology import sliding_window_topology
from src.nullmodel import matched_null, random_cpc_pair_baseline, superposed_epoch
from src.plotting import set_style, save_figure, PALETTE, PALETTE_LIST, year_axis, confidence_band

set_style()
logger = get_logger('nb04')

In [ ]:
# %% Load data and breakthroughs
patents = pd.read_parquet(DATA_DIR / 'patents.parquet')
citations = pd.read_parquet(DATA_DIR / 'citations.parquet')
cpc_map = pd.read_parquet(DATA_DIR / 'cpc_map.parquet')

breakthroughs = load_breakthroughs()
print(f'Breakthroughs: {len(breakthroughs)}')

In [ ]:
# %% Load topology results from Notebook 02 (cached)
import pickle
from pathlib import Path

TOPO_CACHE = DATA_DIR / 'topology_cache'
topology_results = {}

for f in sorted(TOPO_CACHE.glob('sliding_*.pkl')):
    key = f.stem.replace('sliding_', '').split('_w')[0]
    with open(f, 'rb') as fp:
        topology_results[key] = pickle.load(fp)
    print(f'  Loaded {key}: {len(topology_results[key])} windows')

print(f'\nTotal topology results: {len(topology_results)} CPC pairs')

## Compute Matched Null Models

For each breakthrough, generate null topology measurements from the same
CPC pair but in non-breakthrough time periods.

In [ ]:
# %% Generate matched null models
# A100 environment: 500 samples per breakthrough for tighter null distributions
null_results = {}

for bt in tqdm(breakthroughs, desc='Computing null models'):
    null_df = matched_null(
        breakthrough=bt,
        citations=citations,
        cpc_map=cpc_map,
        n_samples=500,
        window_years=5,
        years_before=10,
    )
    null_results[bt.name] = null_df
    
print(f'Null models computed for {len(null_results)} breakthroughs')

## Pre-Breakthrough vs. Null Topology

For each breakthrough: extract β₁ in the 10 years before filing,
compare against the matched null distribution.

In [ ]:
# %% Extract pre-breakthrough metrics and compute z-scores
comparison = []

for bt in breakthroughs:
    # Get topology for this breakthrough's CPC pair
    if len(bt.cpc_sections) >= 2:
        pair_key = f'{bt.cpc_sections[0]}_{bt.cpc_sections[1]}'
    else:
        pair_key = f'{bt.cpc_sections[0]}_{bt.cpc_sections[0]}'
    
    # Try both orderings
    topo = topology_results.get(pair_key)
    if topo is None:
        rev_key = '_'.join(reversed(pair_key.split('_')))
        topo = topology_results.get(rev_key)
    
    if topo is None or len(topo) == 0:
        continue
    
    # Pre-breakthrough window
    start, end = get_precursor_window(bt, years_before=10)
    pre = topo[(topo['year'] >= start) & (topo['year'] <= end)]
    
    if len(pre) == 0:
        continue
    
    pre_beta1 = pre['beta_1'].mean()
    pre_pe = pre['persistence_entropy'].mean()
    
    # Null distribution
    null = null_results.get(bt.name)
    if null is None or len(null) == 0:
        continue
    
    null_beta1_mean = null['beta_1'].mean()
    null_beta1_std = null['beta_1'].std()
    null_pe_mean = null['persistence_entropy'].mean()
    null_pe_std = null['persistence_entropy'].std()
    
    # Z-scores
    z_beta1 = (pre_beta1 - null_beta1_mean) / max(null_beta1_std, 1e-10)
    z_pe = (pre_pe - null_pe_mean) / max(null_pe_std, 1e-10)
    
    comparison.append({
        'name': bt.name,
        'category': bt.category,
        'filing_year': bt.filing_year,
        'pre_beta1': pre_beta1,
        'null_beta1_mean': null_beta1_mean,
        'z_beta1': z_beta1,
        'pre_pe': pre_pe,
        'null_pe_mean': null_pe_mean,
        'z_pe': z_pe,
    })

comp_df = pd.DataFrame(comparison)
print(f'Breakthroughs with valid comparisons: {len(comp_df)}')
print(f'\nMean z-score (β₁):  {comp_df["z_beta1"].mean():.2f} ± {comp_df["z_beta1"].std():.2f}')
print(f'Mean z-score (PE):   {comp_df["z_pe"].mean():.2f} ± {comp_df["z_pe"].std():.2f}')

## Figure 1: Superposed Epoch Plot (THE KEY RESULT)

Align all breakthroughs at t=0 (filing year), average β₁ from -10 to +5 years.
Compare against the null model's 95% confidence interval.

In [ ]:
# %% Figure 1: Superposed epoch analysis
epoch = superposed_epoch(
    breakthroughs=breakthroughs,
    topology_results=topology_results,
    metric='beta_1',
    years_before=10,
    years_after=5,
)

# Generate null epoch (using random baseline)
# A100 environment: 500 samples for tighter confidence intervals
null_baseline = random_cpc_pair_baseline(
    citations=citations,
    cpc_map=cpc_map,
    n_samples=500,
)

fig, ax = plt.subplots(figsize=(12, 7))

if len(epoch) > 0:
    ax.plot(epoch['epoch_year'], epoch['mean'], color=PALETTE['red'], linewidth=2.5,
            label='Pre-breakthrough β₁ (mean)', zorder=5)
    ax.fill_between(
        epoch['epoch_year'],
        epoch['mean'] - epoch['std'],
        epoch['mean'] + epoch['std'],
        alpha=0.2, color=PALETTE['red'], label='±1 SD'
    )

# Null band
if len(null_baseline) > 0:
    null_mean = null_baseline['beta_1'].mean()
    null_std = null_baseline['beta_1'].std()
    ax.axhspan(null_mean - 1.96 * null_std, null_mean + 1.96 * null_std,
               alpha=0.1, color=PALETTE['blue'], label='Null 95% CI')
    ax.axhline(null_mean, color=PALETTE['blue'], linestyle='--', alpha=0.5)

ax.axvline(0, color='gray', linestyle=':', alpha=0.7, label='Breakthrough (t=0)')
ax.set_xlabel('Years Relative to Breakthrough Filing')
ax.set_ylabel('β₁ (Mean Across Breakthroughs)')
ax.set_title('Figure 1: Superposed Epoch Analysis — β₁ Before and After Breakthroughs')
ax.legend()
fig.tight_layout()
save_figure(fig, '04_superposed_epoch')

## Figure 2: Individual Breakthrough β₁ Curves

In [ ]:
# %% Figure 2: Individual breakthrough curves
selected_bts = [bt for bt in breakthroughs if bt.name in [
    'CRISPR-Cas9 Gene Editing', 'PageRank Algorithm', 'Lithium-Ion Battery',
    'Polymerase Chain Reaction (PCR)', 'Multi-Touch Interface (iPhone)',
    'Selective Laser Sintering (SLS) 3D Printing',
    'mRNA Vaccine Platform', 'GPU General-Purpose Computing (CUDA)', 'WiFi (IEEE 802.11)',
]]

n_selected = len(selected_bts)
ncols = 3
nrows = (n_selected + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows), sharex=True)
axes = axes.flatten() if n_selected > 1 else [axes]

for i, bt in enumerate(selected_bts):
    ax = axes[i]
    
    if len(bt.cpc_sections) >= 2:
        pair_key = f'{bt.cpc_sections[0]}_{bt.cpc_sections[1]}'
    else:
        pair_key = f'{bt.cpc_sections[0]}_{bt.cpc_sections[0]}'
    
    topo = topology_results.get(pair_key)
    if topo is None:
        rev_key = '_'.join(reversed(pair_key.split('_')))
        topo = topology_results.get(rev_key)
    
    if topo is not None and len(topo) > 0:
        ax.plot(topo['year'], topo['beta_1'], color=PALETTE['blue'], linewidth=2)
        ax.axvline(bt.filing_year, color=PALETTE['red'], linestyle='--', alpha=0.7, label='Filing year')
        ax.axvspan(bt.filing_year - 10, bt.filing_year, alpha=0.05, color=PALETTE['orange'])
    
    ax.set_title(bt.name, fontsize=10)
    ax.set_ylabel('β₁')

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Figure 2: β₁ Time Series for Individual Breakthroughs', fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, '04_individual_beta1')

## Figure 3: Effect Size Distribution

In [ ]:
# %% Figure 3: Z-score distribution
if len(comp_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # β₁ z-scores
    ax = axes[0]
    ax.barh(range(len(comp_df)), comp_df['z_beta1'].values,
            color=[PALETTE['red'] if z > 0 else PALETTE['blue'] for z in comp_df['z_beta1']])
    ax.set_yticks(range(len(comp_df)))
    ax.set_yticklabels(comp_df['name'].values, fontsize=7)
    ax.axvline(0, color='gray', linestyle='-', alpha=0.5)
    ax.axvline(1.96, color='gray', linestyle='--', alpha=0.3, label='p=0.05')
    ax.axvline(-1.96, color='gray', linestyle='--', alpha=0.3)
    ax.set_xlabel('Z-score (β₁)')
    ax.set_title('β₁ Effect Size')
    ax.legend()
    ax.invert_yaxis()

    # PE z-scores
    ax = axes[1]
    ax.barh(range(len(comp_df)), comp_df['z_pe'].values,
            color=[PALETTE['red'] if z > 0 else PALETTE['blue'] for z in comp_df['z_pe']])
    ax.set_yticks(range(len(comp_df)))
    ax.set_yticklabels(comp_df['name'].values, fontsize=7)
    ax.axvline(0, color='gray', linestyle='-', alpha=0.5)
    ax.axvline(1.96, color='gray', linestyle='--', alpha=0.3)
    ax.axvline(-1.96, color='gray', linestyle='--', alpha=0.3)
    ax.set_xlabel('Z-score (Persistence Entropy)')
    ax.set_title('Persistence Entropy Effect Size')
    ax.invert_yaxis()

    fig.suptitle('Figure 3: Pre-Breakthrough Topology vs. Null Model', fontsize=14, y=1.02)
    fig.tight_layout()
    save_figure(fig, '04_effect_sizes')

## Statistical Tests

In [ ]:
# %% Aggregate statistical tests
if len(comp_df) > 0:
    # One-sample t-test: are z-scores significantly different from 0?
    t_stat_b1, p_val_b1 = stats.ttest_1samp(comp_df['z_beta1'], 0)
    t_stat_pe, p_val_pe = stats.ttest_1samp(comp_df['z_pe'], 0)
    
    print('=== Aggregate Statistical Tests ===')
    print(f'\nβ₁ z-scores: mean={comp_df["z_beta1"].mean():.3f}, t={t_stat_b1:.3f}, p={p_val_b1:.4f}')
    print(f'PE z-scores: mean={comp_df["z_pe"].mean():.3f}, t={t_stat_pe:.3f}, p={p_val_pe:.4f}')
    
    # Wilcoxon signed-rank test (non-parametric)
    try:
        w_stat_b1, wp_b1 = stats.wilcoxon(comp_df['z_beta1'])
        w_stat_pe, wp_pe = stats.wilcoxon(comp_df['z_pe'])
        print(f'\nWilcoxon β₁: W={w_stat_b1:.1f}, p={wp_b1:.4f}')
        print(f'Wilcoxon PE: W={w_stat_pe:.1f}, p={wp_pe:.4f}')
    except ValueError:
        print('Wilcoxon test requires >20 samples, skipped.')
    
    # Effect size (Cohen's d)
    d_b1 = comp_df['z_beta1'].mean() / comp_df['z_beta1'].std() if comp_df['z_beta1'].std() > 0 else 0
    d_pe = comp_df['z_pe'].mean() / comp_df['z_pe'].std() if comp_df['z_pe'].std() > 0 else 0
    print(f'\nCohen\'s d (β₁): {d_b1:.3f}')
    print(f'Cohen\'s d (PE):  {d_pe:.3f}')
    
    # Interpretation
    if p_val_b1 < 0.05:
        print(f'\n** β₁ is significantly different in pre-breakthrough periods (p={p_val_b1:.4f}) **')
    else:
        print(f'\n** β₁ is NOT significantly different from null (p={p_val_b1:.4f}) **')
        print('   This is a null result: topological loops do not systematically precede breakthroughs.')

## Figure 4: ROC-Like Curve

If we treat high β₁ as a "breakthrough detector", how well does it discriminate?

In [ ]:
# %% Figure 4: Detection performance
if len(comp_df) > 0 and len(null_baseline) > 0:
    # Pre-breakthrough β₁ values
    real_values = comp_df['pre_beta1'].values
    null_values = null_baseline['beta_1'].values
    
    # Sweep thresholds
    thresholds = np.linspace(
        min(real_values.min(), null_values.min()),
        max(real_values.max(), null_values.max()),
        100
    )
    
    hit_rates = []
    false_alarm_rates = []
    
    for thresh in thresholds:
        hits = (real_values > thresh).mean()
        false_alarms = (null_values > thresh).mean()
        hit_rates.append(hits)
        false_alarm_rates.append(false_alarms)
    
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(false_alarm_rates, hit_rates, color=PALETTE['red'], linewidth=2)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random (AUC=0.5)')
    
    # Compute AUC
    auc = np.trapz(sorted(hit_rates), sorted(false_alarm_rates))
    ax.set_xlabel('False Alarm Rate')
    ax.set_ylabel('Hit Rate')
    ax.set_title(f'Figure 4: β₁ as Breakthrough Detector (AUC = {abs(auc):.3f})')
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    fig.tight_layout()
    save_figure(fig, '04_roc_curve')

## Summary

**The Precursor Test answers the central question:**
Do topological signatures in the patent citation network systematically
precede technological breakthroughs?

The answer depends on the data. If the superposed epoch shows a clear signal
above the null band, topology is a leading indicator. If not, innovation
is topologically invisible until it arrives — which is itself a meaningful
scientific finding.

Either way, this feeds into Notebook 05 where we test whether topology adds
predictive power beyond simpler metrics.